### Introduction

Below are some of the data processing queries we used during preprocessing. Some additional preprocessing steps, like JSON preprocessing, were executed directly in PostgreSQL and are not fully recorded here.

Please do not re-run the following queries, since they may modify the database and potentially affect the current data.

In [ ]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 91.1 MB/s eta 0:00:00


In [ ]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="cis5500-final.cqrayabndbqk.us-east-1.rds.amazonaws.com",
    port=5432,
    dbname="postgres",
    user="cis5500_final",
    password="final_cis5500"
    )
cur = conn.cursor()

### General Preprocessing of Airport Dataset

In this section, we clean the original airport dataset by removing attributes that do not provide meaningful information for analysis.

Since all airport records are located in the United States, the `COUNTRY` column contains only one unique value (`USA`). Therefore, we remove this column to simplify the schema and reduce unnecessary redundancy.

In [ ]:
query = """
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'airports';
"""

airport_schema = pd.read_sql(query, conn)
airport_schema

In [ ]:
query = """
ALTER TABLE airports
DROP COLUMN country;
"""

conn.execute(query)
conn.commit()

In [ ]:
query = """
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'airports';
"""

updated_airport_schema = pd.read_sql(query, conn)
updated_airport_schema

### General Preprocessing of the Flight Dataset

In this section, we preprocess the original `flight_data_2024` table and create a cleaned `flight` table. This process includes previewing the original schema, removing unnecessary operational columns, renaming attributes for clarity, adding a unique `flight_id` primary key, and verifying the final cleaned schema for normalization and relational querying.

In [ ]:
query = """
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'flight_data_2024';
"""

original_schema = pd.read_sql(query, conn)
original_schema

In [ ]:
query = """
CREATE TABLE flight AS
SELECT
    ROW_NUMBER() OVER () AS flight_id,
    fl_date AS flight_date,
    dep_time,
    weather_delay AS weather_delay_min,
    late_aircraft_delay AS late_aircraft_delay_min,
    cancelled AS is_cancelled,
    origin AS origin_code,
    origin_city_name AS origin_city,
    origin_state_nm AS origin_state
FROM flight_data_2024;
"""

conn.execute(query)
conn.commit()

In [ ]:
query = """
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'flight';
"""

cleaned_schema = pd.read_sql(query, conn)
cleaned_schema

In [ ]:
query = """
ALTER TABLE flight
ADD PRIMARY KEY (flight_id);
"""

conn.execute(query)
conn.commit()

In [ ]:
query = """
SELECT *
FROM flight
LIMIT 10;
"""

flight_preview = pd.read_sql(query, conn)
flight_preview

In [ ]:
query = """
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'flight'
AND column_name IN (
    'month',
    'day_of_month',
    'day_of_week',
    'year',
    'distance',
    'taxi_out',
    'wheels_off',
    'wheels_on',
    'taxi_in',
    'air_time'
);
"""

removed_check = pd.read_sql(query, conn)
removed_check

### Remove Businesses with Zero Reviews

Businesses with zero reviews provide very limited information for rating and popularity analysis. We first computed the number and percentage of businesses with `num_of_reviews = 0` to evaluate their impact on the dataset.

After analysis, we removed these rows from the `rds_businesses` table using a SQL DELETE operation. This preprocessing step helps improve data quality and ensures that later analysis focuses on businesses with meaningful user feedback.

In [ ]:
count_review_zero_query = """
SELECT COUNT(*) AS zero_review_count
FROM rds_businesses
WHERE num_of_reviews = 0;
"""

compute_percentage_review_zero_query = """
SELECT
    COUNT(*) AS zero_review_count,
    ROUND(
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rds_businesses),
        2
    ) AS percentage
FROM rds_businesses
WHERE num_of_reviews = 0;
"""

review_zero_count = pd.read_sql(count_review_zero_query, conn)
review_zero_percentage = pd.read_sql(compute_percentage_review_zero_query, conn)

print(review_zero_count)
print(review_zero_percentage)

In [ ]:
cur.execute("""
DELETE FROM rds_businesses
WHERE num_of_reviews = 0;
""")

print(cur.rowcount)

conn.commit()

### Flight Data Cleaning and Transformation
We transformed the original flight_data_2024 table into a cleaner flight table by:

- Renaming columns (e.g., origin → origin_code, origin_city_name → origin_city)
- Converting data types (e.g., cancelled → boolean, dep_time → time)
- Selecting only relevant attributes for analysis
- Creating a unique flight_id using ROW_NUMBER()

In [ ]:
cur.execute("""
DROP TABLE IF EXISTS flight;

CREATE TABLE flight AS
SELECT
    ROW_NUMBER() OVER () AS flight_id,
    fl_date AS flight_date,
    dep_time::time AS dep_time,
    origin AS origin_code,
    origin_city_name AS origin_city,
    origin_state_nm AS origin_state,
    (cancelled = 1) AS is_cancelled,
    weather_delay AS weather_delay_min,
    late_aircraft_delay AS late_aircraft_delay_min
FROM flight_data_2024;
""")

conn.commit()

### Remove Records with NULL Origin Code for Data Quality and Performance
We computed the percentage of records with NULL origin_code. Since these rows cannot be reliably joined with airport/location tables and do not contribute meaningful information, we exclude them to reduce unnecessary computation and improve query performance.

In [ ]:
query = """
SELECT
    COUNT(*) FILTER (WHERE origin_code IS NULL) AS null_count,
    COUNT(*) AS total_count,
    ROUND(
        COUNT(*) FILTER (WHERE origin_code IS NULL) * 100.0 / COUNT(*),
        4
    ) AS null_percentage
FROM flight_clean;
"""

cur.execute(query)
result = cur.fetchone()

print(result)

In [ ]:
query = """
DELETE FROM flight_clean
WHERE origin_code IS NULL;
"""

cur.execute(query)
print(cur.rowcount)

conn.commit()

### Split the Flights Table for 3NF

To satisfy 3NF, we decomposed the original flight table into two relations. In the original table, origin_city and origin_state depend on origin_code, not directly on flight_id. This creates a transitive dependency: flight_id → origin_code → origin_city, origin_state.

To eliminate this dependency, we separated the airport location information into a new table:

- flight_clean(flight_id, flight_date, dep_time, origin_code, ...)
- origin_locations(origin_code, origin_city, origin_state)

After decomposition, flight_clean only contains flight-specific attributes, while origin_locations stores airport location information. This removes the transitive dependency and ensures both tables satisfy 3NF.

In [ ]:
# 1. Create origin_locations table
cur.execute("""
DROP TABLE IF EXISTS origin_locations;

CREATE TABLE origin_locations (
    origin_code VARCHAR(10) PRIMARY KEY,
    origin_city VARCHAR(100),
    origin_state VARCHAR(10)
);
""")

conn.commit()

In [ ]:
# 2. Insert distinct origin location records
cur.execute("""
INSERT INTO origin_locations (origin_code, origin_city, origin_state)
SELECT DISTINCT
    origin_code,
    origin_city,
    origin_state
FROM flight_clean
WHERE origin_code IS NOT NULL;
""")

conn.commit()

In [ ]:
# 3. Remove transitive-dependent columns from flight_clean
cur.execute("""
ALTER TABLE flight_clean
DROP COLUMN IF EXISTS origin_city,
DROP COLUMN IF EXISTS origin_state;
""")

conn.commit()

### Remove Permanently Closed Stores

When searching for businesses, we should not include stores that are no longer operating. Based on our calculation, permanently closed stores account for about 5.7% of all businesses. Therefore, we removed rows where `status_state = 'Permanently closed'` during preprocessing.

In [ ]:
closed_query = """
SELECT COUNT(*) AS closed_count
FROM rds_businesses
WHERE status_state = 'Permanently closed';
"""

total_query = """
SELECT COUNT(*) AS total_count
FROM rds_businesses;
"""

closed = pd.read_sql(closed_query, conn)
total = pd.read_sql(total_query, conn)

ratio = closed['closed_count'][0] / total['total_count'][0]
percentage = ratio * 100

print(percentage)

/tmp/ipykernel_9964/4204897295.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  closed = pd.read_sql(closed_query, conn)
/tmp/ipykernel_9964/4204897295.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  total = pd.read_sql(total_query, conn)


5.703965361988173


In [ ]:
cur.execute("""
DELETE FROM rds_businesses
WHERE status_state = 'Permanently closed';
""")

print(cur.rowcount)

conn.commit()

### Address to State Transformation for Join Optimization
To improve query performance and reduce join costs, we preprocess the address field to extract the corresponding state for each business.

Since joining large tables on derived location information (e.g., parsing state from address at query time) can be expensive, we materialize the state attribute as part of the rds_businesses table.

This allows us to perform joins directly on the state column, avoiding repeated string parsing and improving efficiency in aggregation queries.

However, for records where the address is missing, incomplete, or does not contain a recognizable state, the extracted state remains NULL. These records cannot be used in state-level joins unless additional geocoding or manual cleaning is applied.

In [ ]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
"""

tables = pd.read_sql(query, conn)
tables

/tmp/ipykernel_3500/219874329.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables = pd.read_sql(query, conn)


,table_name
0,flight_data_2024
1,airports
2,flight_clean
3,rds_businesses
4,rds_categories
5,rds_hours


In [ ]:
query = """
SELECT
    address,
    SUBSTRING(address FROM ',\s*[A-Za-z\s]+,\s*([A-Z]{2})\s+\d{5}$') AS extracted_state
FROM rds_businesses
"""

tables = pd.read_sql(query, conn)
tables

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_3500/2496446702.py:4: SyntaxWarning: invalid escape sequence '\s'
  SUBSTRING(address FROM ',\s*[A-Za-z\s]+,\s*([A-Z]{2})\s+\d{5}$') AS extracted_state
/tmp/ipykernel_3500/2496446702.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables = pd.read_sql(query, conn)


,address,extracted_state
0,"STRATFORD PARK, 530 E 800 N, Orem, UT 84097",UT
1,None,None
2,"Parleys Creek, Utah",None
3,"Wayland Station, 645 E 4065 S, Murray, UT 84107",UT
4,"Murphy Trailhead, Moab, UT 84532",UT
...,...,...
4878566,Logan River,None
4878567,None,None
4878568,"Averett Canyon, Utah",None
4878569,"Chevron, 817 S Main St, Moab, UT 84532",UT


In [ ]:
cur.execute("""
ALTER TABLE rds_businesses
ADD COLUMN state VARCHAR(2);
""")

conn.commit()

In [ ]:
cur.execute("""
UPDATE rds_businesses
SET state = SUBSTRING(address FROM ',\s*[A-Za-z\s]+,\s*([A-Z]{2})\s+\d{5}$');
""")

conn.commit()

In [ ]:
query = """
SELECT *
FROM rds_businesses
"""

tables = pd.read_sql(query, conn)
tables

/tmp/ipykernel_3500/1951532660.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables = pd.read_sql(query, conn)


,gmap_id,name,address,latitude,longitude,avg_rating,num_of_reviews,url,status_state,state
0,0x874d854c8ec760f3:0x96e17af504d16146,STRATFORD PARK,"STRATFORD PARK, 530 E 800 N, Orem, UT 84097",40.311083,-111.682935,3.9,13,https://www.google.com/maps/place//data=!4m2!3...,None,UT
1,0x875303585cd06b6b:0x77ba7ca227159848,Victors Carpet Cleaning,None,40.993094,-111.935667,5.0,8,https://www.google.com/maps/place//data=!4m2!3...,Open 24 hours,None
2,0x875267c6e3c0cd65:0x89796d9d10b447cd,Parleys Creek,"Parleys Creek, Utah",40.741407,-111.740084,4.7,13,https://www.google.com/maps/place/Utah/data=!4...,None,None
3,0x87528a6c22ce2bd5:0xa80b9e573be79c7d,Wayland Station,"Wayland Station, 645 E 4065 S, Murray, UT 84107",40.684241,-111.873065,3.6,8,https://www.google.com/maps/place//data=!4m2!3...,None,UT
4,0x8748195b02685457:0xdd0771433eb09e74,Murphy Trailhead,"Murphy Trailhead, Moab, UT 84532",38.354958,-109.863917,4.7,14,https://www.google.com/maps/place//data=!4m2!3...,None,UT
...,...,...,...,...,...,...,...,...,...,...
4878566,0x875479856abca0b7:0x381a797d92242c80,Logan River,Logan River,41.798136,-111.644789,4.7,34,https://www.google.com/maps/place//data=!4m2!3...,None,None
4878567,0x875261a2f09d0b41:0x9896beaccfa5af4b,Courtland Roofing,None,39.502837,-111.547028,5.0,18,https://www.google.com/maps/place//data=!4m2!3...,Open ⋅ Closes 6PM,None
4878568,0x87351258ee3b4d6d:0xaa929ea641b34da,Averett Canyon,"Averett Canyon, Utah",37.482484,-112.079634,5.0,1,https://www.google.com/maps/place/Utah/data=!4...,None,None
4878569,0x8747e1f511a25f61:0xdce2bd8ae44cd578,Chevron,"Chevron, 817 S Main St, Moab, UT 84532",38.561003,-109.546905,3.4,18,https://www.google.com/maps/place//data=!4m2!3...,Open ⋅ Closes 11PM,UT


In [ ]:
query = """
SELECT
    ROUND(100.0 * SUM(CASE WHEN state IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS state_null_pct
FROM rds_businesses;
"""

pd.read_sql(query, conn)

/tmp/ipykernel_3500/2783998347.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,state_null_pct
0,4.55


In [ ]:
cur = conn.cursor()

cur.execute("""
UPDATE rds_businesses
SET state = COALESCE(
    SUBSTRING(address FROM ',\s*[A-Za-z\s]+,\s*([A-Z]{2})\s+\d{5}(?:-\d{4})?$'),
    CASE
        WHEN address ILIKE '%Alabama%' THEN 'AL'
        WHEN address ILIKE '%Alaska%' THEN 'AK'
        WHEN address ILIKE '%Arizona%' THEN 'AZ'
        WHEN address ILIKE '%Arkansas%' THEN 'AR'
        WHEN address ILIKE '%California%' THEN 'CA'
        WHEN address ILIKE '%Colorado%' THEN 'CO'
        WHEN address ILIKE '%Connecticut%' THEN 'CT'
        WHEN address ILIKE '%Delaware%' THEN 'DE'
        WHEN address ILIKE '%Florida%' THEN 'FL'
        WHEN address ILIKE '%Georgia%' THEN 'GA'
        WHEN address ILIKE '%Hawaii%' THEN 'HI'
        WHEN address ILIKE '%Idaho%' THEN 'ID'
        WHEN address ILIKE '%Illinois%' THEN 'IL'
        WHEN address ILIKE '%Indiana%' THEN 'IN'
        WHEN address ILIKE '%Iowa%' THEN 'IA'
        WHEN address ILIKE '%Kansas%' THEN 'KS'
        WHEN address ILIKE '%Kentucky%' THEN 'KY'
        WHEN address ILIKE '%Louisiana%' THEN 'LA'
        WHEN address ILIKE '%Maine%' THEN 'ME'
        WHEN address ILIKE '%Maryland%' THEN 'MD'
        WHEN address ILIKE '%Massachusetts%' THEN 'MA'
        WHEN address ILIKE '%Michigan%' THEN 'MI'
        WHEN address ILIKE '%Minnesota%' THEN 'MN'
        WHEN address ILIKE '%Mississippi%' THEN 'MS'
        WHEN address ILIKE '%Missouri%' THEN 'MO'
        WHEN address ILIKE '%Montana%' THEN 'MT'
        WHEN address ILIKE '%Nebraska%' THEN 'NE'
        WHEN address ILIKE '%Nevada%' THEN 'NV'
        WHEN address ILIKE '%New Hampshire%' THEN 'NH'
        WHEN address ILIKE '%New Jersey%' THEN 'NJ'
        WHEN address ILIKE '%New Mexico%' THEN 'NM'
        WHEN address ILIKE '%New York%' THEN 'NY'
        WHEN address ILIKE '%North Carolina%' THEN 'NC'
        WHEN address ILIKE '%North Dakota%' THEN 'ND'
        WHEN address ILIKE '%Ohio%' THEN 'OH'
        WHEN address ILIKE '%Oklahoma%' THEN 'OK'
        WHEN address ILIKE '%Oregon%' THEN 'OR'
        WHEN address ILIKE '%Pennsylvania%' THEN 'PA'
        WHEN address ILIKE '%Rhode Island%' THEN 'RI'
        WHEN address ILIKE '%South Carolina%' THEN 'SC'
        WHEN address ILIKE '%South Dakota%' THEN 'SD'
        WHEN address ILIKE '%Tennessee%' THEN 'TN'
        WHEN address ILIKE '%Texas%' THEN 'TX'
        WHEN address ILIKE '%Utah%' THEN 'UT'
        WHEN address ILIKE '%Vermont%' THEN 'VT'
        WHEN address ILIKE '%Virginia%' THEN 'VA'
        WHEN address ILIKE '%Washington%' THEN 'WA'
        WHEN address ILIKE '%West Virginia%' THEN 'WV'
        WHEN address ILIKE '%Wisconsin%' THEN 'WI'
        WHEN address ILIKE '%Wyoming%' THEN 'WY'
        WHEN address ILIKE '%District of Columbia%' THEN 'DC'
        ELSE state
    END
)
WHERE state IS NULL;
""")

conn.commit()

<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_3500/2353309042.py:6: SyntaxWarning: invalid escape sequence '\s'
  SUBSTRING(address FROM ',\s*[A-Za-z\s]+,\s*([A-Z]{2})\s+\d{5}(?:-\d{4})?$'),


In [ ]:
query = """
SELECT
    ROUND(100.0 * SUM(CASE WHEN state IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS state_null_pct
FROM rds_businesses;
"""

pd.read_sql(query, conn)

/tmp/ipykernel_3500/2783998347.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,state_null_pct
0,3.67


In [ ]:
query = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'rds_businesses';
"""

pd.read_sql(query, conn)


/tmp/ipykernel_3500/4037201249.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,column_name,data_type
0,num_of_reviews,integer
1,latitude,double precision
2,longitude,double precision
3,avg_rating,double precision
4,status_state,text
5,gmap_id,text
6,state,character varying
7,name,text
8,address,text
9,url,text


In [ ]:
query = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'rds_categories';
"""

pd.read_sql(query, conn)

/tmp/ipykernel_3500/1684550038.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,column_name,data_type
0,gmap_id,text
1,category_name,text


In [ ]:
query = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'rds_hours';
"""

pd.read_sql(query, conn)

/tmp/ipykernel_3500/947210860.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,column_name,data_type
0,gmap_id,text
1,day,text
2,hours_text,text


### Remove Businesses with Missing Names

Businesses without names are incomplete entities and cannot be meaningfully displayed, searched, or analyzed. Therefore, we removed rows where the `name` attribute was NULL during preprocessing.

In [ ]:
count_null_name_query = """
SELECT COUNT(*) AS null_name_count
FROM rds_businesses
WHERE name IS NULL;
"""

compute_percentage_null_name_query = """
SELECT
    COUNT(*) AS null_name_count,
    ROUND(
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rds_businesses),
        2
    ) AS percentage
FROM rds_businesses
WHERE name IS NULL;
"""

null_name_count = pd.read_sql(count_null_name_query, conn)
null_name_percentage = pd.read_sql(compute_percentage_null_name_query, conn)

print(null_name_count)
print(null_name_percentage)

In [ ]:
delete_null_name_query = """
DELETE FROM rds_businesses
WHERE name IS NULL;
"""

cur.execute(delete_null_name_query)

print(cur.rowcount)

conn.commit()